### This is a basic tutorial on how to inspect reaktoro states and solver when it fails to solve or initialize. 

In [1]:
## Import core components
from pyomo.environ import (
    Var,
    ConcreteModel,
    units as pyunits,
)

# Ideas core components
from idaes.core import FlowsheetBlock

# Import reaktoro-pse and reaktoro
from reaktoro_pse.reaktoro_block import ReaktoroBlock
import reaktoro as rkt


#### Lets build basic sea water example

In [2]:
sea_water_composition = {
    "Na": 10556,
    "K": 380,
    "Ca": 400,
    "Mg": 1262,
    "Cl": 18978,
    "SO4": 2649,
    "HCO3": 140,
}
# Get ions
ions = list(sea_water_composition.keys())
#

sea_water_ph = 7.56
mass_flow_dict={ion:value/1e6*pyunits.kg/pyunits.s for ion, value in sea_water_composition.items()}
mass_flow_dict['H2O']=1*pyunits.kg/pyunits.s
m = ConcreteModel()
# create IDAES flowsheet
m.fs = FlowsheetBlock(dynamic=False)
m.fs.mass_flow=Var(mass_flow_dict.keys(), initialize=lambda m, i: mass_flow_dict[i], units=pyunits.kg/pyunits.s)
m.fs.mass_flow.fix()
m.fs.pH=Var(initialize=sea_water_ph, units=pyunits.dimensionless)
m.fs.pH.fix()
m.fs.temperature=Var(initialize=293, units=pyunits.K)
m.fs.pressure=Var(initialize=101325, units=pyunits.Pa)
m.fs.H_modifier=Var(initialize=0, units=pyunits.mol/pyunits.s)
# can use NaCl or Sea water prop pack

### Basic reaktoro block with speciation
We are adding H modifier as an example of basic chemical addition which will break reaktoro solver. In practice you add an acid or base that is charge balanced (e.g. HCl, or NaOH)

In [3]:
m.fs.reaktoro_outputs = Var(
        [("scalingTendency", "Calcite"), ("pH", None)],
        initialize=1,
    )
m.fs.eq_precipitation = ReaktoroBlock(
    aqueous_phase={
        "composition": m.fs.mass_flow,  # This is the speciesmass flow
        "convert_to_rkt_species": True,  # We can use default converter as its defined for default database (Phreeqc and pitzer)
        "activity_model": rkt.ActivityModelPitzer(),  # Can provide a string, or Reaktoro initialized class
        "fixed_solvent_specie": "H2O",  # We need to define our aqueous solvent as we have to speciate the block
    },
    system_state={
        "temperature": m.fs.temperature,
        "pressure": m.fs.pressure,
        "pH": m.fs.pH,
    },
    mineral_phase={"phase_components": ["Calcite"]},
    chemistry_modifier={
        "H": m.fs.H_modifier,
    },
    outputs=m.fs.reaktoro_outputs,  # This is the output dictionary we defined above
    database_file="pitzer.dat",  # needs to be a string that names the database file or points to its location
    build_speciation_block=True,  # This will build the constraints to speciate the block
    assert_charge_neutrality_on_property_block=False,
)

2026-03-06 13:06:33 [INFO] idaes.reaktoro_pse.core.reaktoro_inputs: Exact speciation is not provided! Fixing aqueous solvent and, excluding H
2026-03-06 13:06:33 [INFO] idaes.reaktoro_pse.core.reaktoro_inputs: Exact speciation is not provided! Fixing aqueous solvent and, excluding O
2026-03-06 13:06:33 [INFO] idaes.reaktoro_pse.core.reaktoro_solver: rktSolver inputs: ['[Cl]', '[C]', '[Na]', '[Mg]', '[S]', '[K]', '[Ca]', '[H2O]', '[H]', '[O]', '[H+]']
2026-03-06 13:06:33 [INFO] idaes.reaktoro_pse.core.reaktoro_solver: rktSolver constraints: ['charge', 'C_constraint', 'Na_constraint', 'Mg_constraint', 'S_constraint', 'K_constraint', 'Ca_constraint', 'H2O_constraint', 'H_dummy_constraint', 'O_dummy_constraint', 'pH']
2026-03-06 13:06:34 [INFO] idaes.reaktoro_pse.core.reaktoro_solver: rktSolver inputs: ['[H]', '[C]', '[O]', '[Na]', '[Mg]', '[S]', '[Cl]', '[K]', '[Ca]']
2026-03-06 13:06:34 [INFO] idaes.reaktoro_pse.core.reaktoro_solver: rktSolver constraints: ['H_constraint', 'C_constraint'

### Lets access base reaktro state created by reaktoro-pse
Reaktoro-pse will create a standard reaktoro state object which is then used for calculations.

The basic process is:
 1) create rekatoro state
 2) create constraints that convert input species into element amounts
 3) create a reaktoro solver object that enables us to solve the state with coupled constraints

When intialaizing reaktoro-pse block we do following steps:
 1) Equalibrate the state using given composition, temperature, and pressure, ignoring pH, pE.
 2) Fix input values into constraints (e.g. element or species amounts, pH, pE)
 3) Solve the constrained equilibrium problem 
 4) Get derivatives and output values with respect to our inputs (Not covered or shown in this demo)

### Lets first get the base state

In [4]:
# lets get base configurations from our configured reaktoro block
reaktoro_block=m.fs.eq_precipitation

# get standard state we configured!
speciation_reaktoro_state=m.fs.eq_precipitation.speciation_block.rkt_state
speciation_reaktoro_state.set_rkt_state() # This will update the values to once we specified in our inputs
print(speciation_reaktoro_state.state)

+-----------------+-------------+------+
| Property        |       Value | Unit |
+-----------------+-------------+------+
| Temperature     |    293.0000 |    K |
| Pressure        |      1.0132 |  bar |
| Charge:         | -3.3825e-05 |  mol |
| Element Amount: |             |      |
| :: H            |  1.1101e+02 |  mol |
| :: C            |  2.2943e-03 |  mol |
| :: O            |  5.5623e+01 |  mol |
| :: Na           |  4.5917e-01 |  mol |
| :: Mg           |  5.1926e-02 |  mol |
| :: S            |  2.7575e-02 |  mol |
| :: Cl           |  5.3529e-01 |  mol |
| :: K            |  9.7192e-03 |  mol |
| :: Ca           |  9.9803e-03 |  mol |
| Species Amount: |             |      |
| :: H+           |  1.0000e-16 |  mol |
| :: H2O          |  5.5506e+01 |  mol |
| :: CO3-2        |  1.0000e-16 |  mol |
| :: CO2          |  1.0000e-16 |  mol |
| :: Ca+2         |  9.9803e-03 |  mol |
| :: Cl-          |  5.3529e-01 |  mol |
| :: HCO3-        |  2.2943e-03 |  mol |
| :: SO4-2      

During reaktoro block initialization we first equilibrate the state in the output above you can see that many species are at 1e-16, they we created but have not be equilibrated yet

In [5]:
rkt.equilibrate(speciation_reaktoro_state.state)
print(speciation_reaktoro_state.state)

+-----------------+-------------+------+
| Property        |       Value | Unit |
+-----------------+-------------+------+
| Temperature     |    293.0000 |    K |
| Pressure        |      1.0132 |  bar |
| Charge:         | -3.3825e-05 |  mol |
| Element Amount: |             |      |
| :: H            |  1.1101e+02 |  mol |
| :: C            |  2.2943e-03 |  mol |
| :: O            |  5.5623e+01 |  mol |
| :: Na           |  4.5917e-01 |  mol |
| :: Mg           |  5.1926e-02 |  mol |
| :: S            |  2.7575e-02 |  mol |
| :: Cl           |  5.3529e-01 |  mol |
| :: K            |  9.7192e-03 |  mol |
| :: Ca           |  9.9803e-03 |  mol |
| Species Amount: |             |      |
| :: H+           |  2.5085e-08 |  mol |
| :: H2O          |  5.5506e+01 |  mol |
| :: CO3-2        |  3.0843e-05 |  mol |
| :: CO2          |  5.9631e-05 |  mol |
| :: Ca+2         |  9.9803e-03 |  mol |
| :: Cl-          |  5.3529e-01 |  mol |
| :: HCO3-        |  2.1763e-03 |  mol |
| :: SO4-2      

As you can see its equilibrated, but not charged balanced, actually this is the most basic speciation calculations we can do, there are no constraints to balance charge or even pH for example if we get aqueous properties and note that pH is not even what we set it to just yet, nor is charge balanced.

In [6]:
print(rkt.AqueousProps(speciation_reaktoro_state.state))

+-------------------------------------+-------------+-------+
| Property                            |       Value |  Unit |
+-------------------------------------+-------------+-------+
| Temperature                         |    293.0000 |     K |
| Pressure                            |      1.0132 |   bar |
| Ionic Strength (Effective)          |      0.6821 | molal |
| Ionic Strength (Stoichiometric)     |      0.6823 | molal |
| pH                                  |      7.7324 |       |
| pE                                  |    -10.7296 |       |
| Eh                                  |     -0.6238 |     V |
| Alkalinity                          |      0.0022 |  eq/L |
| Charge Molality                     | -3.3825e-05 | molal |
| Element Molality:                   |             |       |
| :: C                                |  2.2943e-03 | molal |
| :: Na                               |  4.5917e-01 | molal |
| :: Mg                               |  5.1926e-02 | molal |
| :: S  

In reaktoro-pse we must do something a-bit more complicated. We need to write a problem with constraints where we set the pH, charge balance, and also enforce elemental balance (so we can change composition on the fly and get derivatives as well)

These objects are available in the reaktoro block, specifically, we will focus on reaktoro inputs, and grab equilibrium specs, which define all of our constraints and variables for our problem

In [7]:
rkt_input_spec= reaktoro_block.speciation_block.rkt_inputs
for key, ls in rkt_input_spec.constraint_dict.items():
    print(f"{key}: {ls}")

# this defines our equilibrium specification 
speciation_equilibrium_specs = rkt_input_spec.equilibrium_specs
print(speciation_equilibrium_specs)

# get existing variables 
existing_variables = speciation_equilibrium_specs.namesControlVariables()

# note the inputs for elements and H+ which is used to enforce our pH constraint
print(existing_variables)
existing_constraints = speciation_equilibrium_specs.namesConstraints()

# note how we have a constraint for charge, and each element except Cl As its used
# for charge balance (e.g. it is free) 
# also note we set the H and O dummy constraints, these are there to ensure we have as many constarints as variables, even though amouunt of O and H is 
# not fixed and only H2O is fixed. 
print(existing_constraints)

C: [(1.0, 'HCO3-')]
Na: [(1.0, 'Na+')]
Mg: [(1.0, 'Mg+2')]
S: [(1.0, 'SO4-2')]
K: [(1.0, 'K+')]
Ca: [(1.0, 'Ca+2')]
['[Cl]', '[C]', '[Na]', '[Mg]', '[S]', '[K]', '[Ca]', '[H2O]', '[H]', '[O]', '[H+]']
['charge', 'C_constraint', 'Na_constraint', 'Mg_constraint', 'S_constraint', 'K_constraint', 'Ca_constraint', 'H2O_constraint', 'H_dummy_constraint', 'O_dummy_constraint', 'pH']


We now can use these objects to build an equlibrum solver problems with reaktoro - this is basically the same thing we do in ReaktoroSolver class.

In [8]:
# lets manually setup a solver and use our specs object to solve a reaktoro problem
solver = rkt.EquilibriumSolver(speciation_equilibrium_specs)
conditions = rkt.EquilibriumConditions(speciation_equilibrium_specs)

# lets edit solver options
solver_options = rkt.EquilibriumOptions()
solver_options.epsilon = 1e-32
solver_options.optima.maxiters=200
solver_options.optima.convergence.tolerance=1e-8
solver.setOptions(solver_options)

Now we can specify our pH, charge, and other inputs!

In [9]:
# Lets configure our inputs!
conditions.set("pH", 7.56)
conditions.set("charge", 0)
conditions.temperature(293,'K')
conditions.pressure(101325,'Pa')

# we also can access intial specie composition on pyomo model through rkt_inputs.
for ion, input_object in rkt_input_spec.rkt_inputs.items():
    if input_object.get_rkt_input_name() not in ['pH','temperature','pressure','charge']:
        print(f'Ion name based on RKT notation: {ion}, rkt name in equalbirium specs: {input_object.get_rkt_input_name()}')        
        mols=input_object.get_value(apply_conversion=True)
        conditions.set(input_object.get_rkt_input_name(), mols)
# Note - the rkt_input_object is tied to input variables for example
original=rkt_input_spec.rkt_inputs['Ca+2'].get_value(apply_conversion=True)
m.fs.mass_flow['Ca'].fix(m.fs.mass_flow['Ca']*2)
print('original Ca value:',original, 'updated Ca value:', rkt_input_spec.rkt_inputs['Ca+2'].get_value(apply_conversion=True))
# Lets solve our problem
result=solver.solve(speciation_reaktoro_state.state, conditions)
print('solve returned successful', result.succeeded())
assert result.succeeded()

Ion name based on RKT notation: HCO3-, rkt name in equalbirium specs: inputHCO3-
Ion name based on RKT notation: Na+, rkt name in equalbirium specs: inputNa+
Ion name based on RKT notation: Mg+2, rkt name in equalbirium specs: inputMg+2
Ion name based on RKT notation: SO4-2, rkt name in equalbirium specs: inputSO4-2
Ion name based on RKT notation: K+, rkt name in equalbirium specs: inputK+
Ion name based on RKT notation: Ca+2, rkt name in equalbirium specs: inputCa+2
Ion name based on RKT notation: H2O, rkt name in equalbirium specs: H2O
original Ca value: 0.009980313123715771 updated Ca value: 0.019960626247431543
solve returned successful True


Now we we shall see that we are charge balanced, the system also reached our target pH.

In [10]:
# check equilibrated state with our constraints
print(speciation_reaktoro_state.state)
print(rkt.AqueousProps(speciation_reaktoro_state.state))

# Note how the pH matches to what we set and our charge is now zero!

+-----------------+-------------+------+
| Property        |       Value | Unit |
+-----------------+-------------+------+
| Temperature     |    293.0000 |    K |
| Pressure        |      1.0132 |  bar |
| Charge:         | -1.2923e-16 |  mol |
| Element Amount: |             |      |
| :: H            |  1.1101e+02 |  mol |
| :: C            |  2.2943e-03 |  mol |
| :: O            |  5.5623e+01 |  mol |
| :: Na           |  4.5917e-01 |  mol |
| :: Mg           |  5.1926e-02 |  mol |
| :: S            |  2.7575e-02 |  mol |
| :: Cl           |  5.3531e-01 |  mol |
| :: K            |  9.7192e-03 |  mol |
| :: Ca           |  9.9803e-03 |  mol |
| Species Amount: |             |      |
| :: H+           |  3.7311e-08 |  mol |
| :: H2O          |  5.5506e+01 |  mol |
| :: CO3-2        |  2.0648e-05 |  mol |
| :: CO2          |  8.8312e-05 |  mol |
| :: Ca+2         |  9.9803e-03 |  mol |
| :: Cl-          |  5.3531e-01 |  mol |
| :: HCO3-        |  2.1669e-03 |  mol |
| :: SO4-2      

We can now get our property block, in reaktoro-pse to change an output state, we need to define a property block, where we pass in exact amount of species from our speciation blcok, as well as modify the state (either add chemicals, or change state variable such as temperature or pressure)

In [11]:
# Lets now do our property block, same as aobve get our state and get all variables
prop_reaktoro_state=reaktoro_block.rkt_state

prop_rkt_input_spec= reaktoro_block.rkt_inputs

# we can inspect the constraitns, you can see how modifer_H shows up for total H amount!
for key, ls in prop_rkt_input_spec.constraint_dict.items():
    print(f"{key}: {ls}")

prop_equilibrium_specs = prop_rkt_input_spec.equilibrium_specs
print(prop_equilibrium_specs)
print(prop_equilibrium_specs.namesControlVariables())
print(prop_equilibrium_specs.namesConstraints())


H: [(1.0, 'H+'), (2.0, 'H2O'), (1.0, 'HCO3-'), (1.0, 'HSO4-'), (1.0, 'MgOH+'), (1.0, 'OH-'), (1, 'modifier_H')]
C: [(1.0, 'CO3-2'), (1.0, 'CO2'), (1.0, 'HCO3-'), (1.0, 'MgCO3')]
O: [(1.0, 'H2O'), (3.0, 'CO3-2'), (2.0, 'CO2'), (3.0, 'HCO3-'), (4.0, 'SO4-2'), (4.0, 'HSO4-'), (3.0, 'MgCO3'), (1.0, 'MgOH+'), (1.0, 'OH-')]
Na: [(1.0, 'Na+')]
Mg: [(1.0, 'Mg+2'), (1.0, 'MgCO3'), (1.0, 'MgOH+')]
S: [(1.0, 'SO4-2'), (1.0, 'HSO4-')]
Cl: [(1.0, 'Cl-')]
K: [(1.0, 'K+')]
Ca: [(1.0, 'Ca+2')]
['[H]', '[C]', '[O]', '[Na]', '[Mg]', '[S]', '[Cl]', '[K]', '[Ca]']
['H_constraint', 'C_constraint', 'O_constraint', 'Na_constraint', 'Mg_constraint', 'S_constraint', 'Cl_constraint', 'K_constraint', 'Ca_constraint']


In [12]:
print(prop_reaktoro_state.state)

# what you will also note, no inputs have been actually updated
# its because we normally pass the true mol amounts from our property state into this block, lets do that  next

+---------------------+------------+------+
| Property            |      Value | Unit |
+---------------------+------------+------+
| Temperature         |   293.0000 |    K |
| Pressure            |     1.0132 |  bar |
| Charge:             | 0.0000e+00 |  mol |
| Element Amount:     |            |      |
| :: H                | 2.0005e+01 |  mol |
| :: C                | 4.0000e-03 |  mol |
| :: O                | 1.0021e+01 |  mol |
| :: Na               | 1.0000e-03 |  mol |
| :: Mg               | 3.0000e-03 |  mol |
| :: S                | 2.0000e-03 |  mol |
| :: Cl               | 1.0000e-03 |  mol |
| :: K                | 1.0000e-03 |  mol |
| :: Ca               | 1.0000e-03 |  mol |
| Species Amount:     |            |      |
| :: H+               | 1.0000e-03 |  mol |
| :: H2O              | 1.0000e+01 |  mol |
| :: CO3-2            | 1.0000e-03 |  mol |
| :: CO2              | 1.0000e-03 |  mol |
| :: Ca+2             | 1.0000e-03 |  mol |
| :: Cl-              | 1.0000e-

We can get true species amount from our speciation block, and set them on our property block as shown below 

In [13]:
for phase in prop_reaktoro_state.inputs.registered_phases:
    for species in prop_reaktoro_state.inputs.species_list[phase]:
        if species in prop_reaktoro_state.inputs:
            val= speciation_reaktoro_state.state.speciesAmount(species)
            prop_reaktoro_state.state.set(species, val,'mol')
            print(species, val)


H+ 3.73107e-08
H2O 55.5062
CO3-2 2.06476e-05
CO2 8.8312e-05
Ca+2 0.00998031
Cl- 0.535307
HCO3- 0.00216692
SO4-2 0.027575
HSO4- 8.64184e-09
Mg+2 0.0519069
K+ 0.00971923
MgCO3 1.84642e-05
MgOH+ 4.22224e-07
Na+ 0.459171
OH- 4.13443e-07


In [14]:
rkt.equilibrate(prop_reaktoro_state.state)
print(prop_reaktoro_state.state)

+---------------------+------------+------+
| Property            |      Value | Unit |
+---------------------+------------+------+
| Temperature         |   293.0000 |    K |
| Pressure            |     1.0132 |  bar |
| Charge:             | 2.3643e-12 |  mol |
| Element Amount:     |            |      |
| :: H                | 1.1101e+02 |  mol |
| :: C                | 2.2943e-03 |  mol |
| :: O                | 5.5623e+01 |  mol |
| :: Na               | 4.5917e-01 |  mol |
| :: Mg               | 5.1926e-02 |  mol |
| :: S                | 2.7575e-02 |  mol |
| :: Cl               | 5.3531e-01 |  mol |
| :: K                | 9.7192e-03 |  mol |
| :: Ca               | 9.9803e-03 |  mol |
| Species Amount:     |            |      |
| :: H+               | 3.7311e-08 |  mol |
| :: H2O              | 5.5506e+01 |  mol |
| :: CO3-2            | 2.0648e-05 |  mol |
| :: CO2              | 8.8312e-05 |  mol |
| :: Ca+2             | 9.9803e-03 |  mol |
| :: Cl-              | 5.3531e-

Now we can solve our equilibrium problem where we enforce the full elemental balance and state. Although we normally could get away with above equilibrate state method, we need to solve the equilibrium problem to get out our derivatives for use in ipopt- this is not shown here. 

In [15]:
# now lets set up our reaction configuration 
# lets manually setup a solver and use our specs object to solve a reaktoro problem
solver_props = rkt.EquilibriumSolver(prop_equilibrium_specs)
conditions_props = rkt.EquilibriumConditions(prop_equilibrium_specs)
solver_props.setOptions(solver_options)


In [16]:
# Lets configure our inputs!
conditions_props.temperature(293,'K')
conditions_props.pressure(101325,'Pa')
for ion, input_object in prop_rkt_input_spec.rkt_inputs.items():
    if input_object.get_rkt_input_name() not in ['pH','temperature','pressure','charge']:
        print(f'Ion name based on RKT notation: {ion}, rkt name in equalbirium specs: {input_object.get_rkt_input_name()}')        
        if 'modifier' in input_object.get_rkt_input_name():
            print('skipping modifier')
            continue
        # mols=input_object.get_value(apply_conversion=True)
        mols= speciation_reaktoro_state.state.speciesAmount(ion) # grab this from our speciation state which is now updated with our input values
        conditions_props.set(input_object.get_rkt_input_name(), mols)
H_modifier_name=prop_rkt_input_spec.rkt_inputs['modifier_H'].get_rkt_input_name()
conditions_props.set(H_modifier_name, 1e-8)
# Note - the rkt_input_object is tied to input variables for example
# Lets solve our problem
result=solver_props.solve(prop_reaktoro_state.state, conditions_props)
print('solve returned successful', result.succeeded())
assert result.succeeded()

Ion name based on RKT notation: H+, rkt name in equalbirium specs: inputH+
Ion name based on RKT notation: H2O, rkt name in equalbirium specs: inputH2O
Ion name based on RKT notation: HCO3-, rkt name in equalbirium specs: inputHCO3-
Ion name based on RKT notation: HSO4-, rkt name in equalbirium specs: inputHSO4-
Ion name based on RKT notation: MgOH+, rkt name in equalbirium specs: inputMgOH+
Ion name based on RKT notation: OH-, rkt name in equalbirium specs: inputOH-
Ion name based on RKT notation: modifier_H, rkt name in equalbirium specs: inputmodifier_H
skipping modifier
Ion name based on RKT notation: CO3-2, rkt name in equalbirium specs: inputCO3-2
Ion name based on RKT notation: CO2, rkt name in equalbirium specs: inputCO2
Ion name based on RKT notation: MgCO3, rkt name in equalbirium specs: inputMgCO3
Ion name based on RKT notation: SO4-2, rkt name in equalbirium specs: inputSO4-2
Ion name based on RKT notation: Na+, rkt name in equalbirium specs: inputNa+
Ion name based on RKT 

In [17]:
# review the state
print(prop_reaktoro_state.state)
print(rkt.AqueousProps(prop_reaktoro_state.state))

+---------------------+------------+------+
| Property            |      Value | Unit |
+---------------------+------------+------+
| Temperature         |   293.0000 |    K |
| Pressure            |     1.0132 |  bar |
| Charge:             | 4.7286e-12 |  mol |
| Element Amount:     |            |      |
| :: H                | 1.1101e+02 |  mol |
| :: C                | 2.2943e-03 |  mol |
| :: O                | 5.5623e+01 |  mol |
| :: Na               | 4.5917e-01 |  mol |
| :: Mg               | 5.1926e-02 |  mol |
| :: S                | 2.7575e-02 |  mol |
| :: Cl               | 5.3531e-01 |  mol |
| :: K                | 9.7192e-03 |  mol |
| :: Ca               | 9.9803e-03 |  mol |
| Species Amount:     |            |      |
| :: H+               | 3.7310e-08 |  mol |
| :: H2O              | 5.5506e+01 |  mol |
| :: CO3-2            | 2.0648e-05 |  mol |
| :: CO2              | 8.8310e-05 |  mol |
| :: Ca+2             | 9.9803e-03 |  mol |
| :: Cl-              | 5.3531e-

Great lets now increase amount of H+ we want to add and we will track the solver progress by enablign `optima.output` which will create `optima.log.txt` which can be used to see the progress of reaktoro solver

In [18]:
modifier_name=prop_rkt_input_spec.rkt_inputs['modifier_H'].get_rkt_input_name()
conditions_props.set(modifier_name, 10e-8)
# Note - the rkt_input_object is tied to input variables for example
# Lets solve our problem

solver_options.optima.convergence.tolerance=1e-8
solver_options.optima.output.active = True
solver_props.setOptions(solver_options)
result=solver_props.solve(prop_reaktoro_state.state, conditions_props)

print('solve returned successful', result.succeeded())
print(prop_reaktoro_state.state)

print(rkt.AqueousProps(prop_reaktoro_state.state))

solve returned successful False
+---------------------+------------+------+
| Property            |      Value | Unit |
+---------------------+------------+------+
| Temperature         |   293.0000 |    K |
| Pressure            |     1.0132 |  bar |
| Charge:             | 7.0929e-12 |  mol |
| Element Amount:     |            |      |
| :: H                | 1.1101e+02 |  mol |
| :: C                | 2.2943e-03 |  mol |
| :: O                | 5.5623e+01 |  mol |
| :: Na               | 4.5917e-01 |  mol |
| :: Mg               | 5.1926e-02 |  mol |
| :: S                | 2.7575e-02 |  mol |
| :: Cl               | 5.3531e-01 |  mol |
| :: K                | 9.7192e-03 |  mol |
| :: Ca               | 9.9803e-03 |  mol |
| Species Amount:     |            |      |
| :: H+               | 3.7301e-08 |  mol |
| :: H2O              | 5.5506e+01 |  mol |
| :: CO3-2            | 2.0653e-05 |  mol |
| :: CO2              | 8.8289e-05 |  mol |
| :: Ca+2             | 9.9803e-03 |  mol |


The above solve failed. We can inspect the `optimal.log.txt` and see that the "Error" could not be reduced to below `1.66e-8`. We can adjust a range of options for optima, but geneneraly relaxing the solver is easiest way, lets reduce to the `convergence.tolerance` to 1e-7. Although H+ additon should solve, its not really practical, as we always add a charge balanced reactant, suc has HCl/NaOH etc.

In [19]:
modifier_name=prop_rkt_input_spec.rkt_inputs['modifier_H'].get_rkt_input_name()
conditions_props.set(modifier_name, 10e-8)
# Note - the rkt_input_object is tied to input variables for example
# Lets solve our problem
solver_options.optima.output.active = True

solver_options.optima.backtracksearch.apply_min_max_fix_and_accept = True
solver_options.optima.convergence.tolerance=1e-7 # ONLY CHANGED THIS!
solver_options.logarithm_barrier_factor=100
solver_options.epsilon = 1e-16
solver_options.optima.linesearch.tolerance =1e-5
solver_options.optima.linesearch.trigger_when_current_error_is_greater_than_initial_error_by_factor =1
solver_options.optima.linesearch.trigger_when_current_error_is_greater_than_previous_error_by_factor  =2
solver_options.optima.steepestdescent.tolerance = 1e-85
solver_options.optima.steepestdescent.maxiters =  20
solver_options.optima.maxiters=50
solver_props.setOptions(solver_options)

result=solver_props.solve(prop_reaktoro_state.state, conditions_props)

print('solve returned successful', result.succeeded())
print(prop_reaktoro_state.state)

print(rkt.AqueousProps(prop_reaktoro_state.state))

solve returned successful True
+---------------------+------------+------+
| Property            |      Value | Unit |
+---------------------+------------+------+
| Temperature         |   293.0000 |    K |
| Pressure            |     1.0132 |  bar |
| Charge:             | 9.4572e-12 |  mol |
| Element Amount:     |            |      |
| :: H                | 1.1101e+02 |  mol |
| :: C                | 2.2943e-03 |  mol |
| :: O                | 5.5623e+01 |  mol |
| :: Na               | 4.5917e-01 |  mol |
| :: Mg               | 5.1926e-02 |  mol |
| :: S                | 2.7575e-02 |  mol |
| :: Cl               | 5.3531e-01 |  mol |
| :: K                | 9.7192e-03 |  mol |
| :: Ca               | 9.9803e-03 |  mol |
| Species Amount:     |            |      |
| :: H+               | 3.7301e-08 |  mol |
| :: H2O              | 5.5506e+01 |  mol |
| :: CO3-2            | 2.0653e-05 |  mol |
| :: CO2              | 8.8289e-05 |  mol |
| :: Ca+2             | 9.9803e-03 |  mol |
|

Note how above solved, but the pH basically did not change, this suggests our problem did not actually solve for new equlbrium, rather we did not really solve anything, as adding pure H+ is impossible in this case with out some sort of counter ion. 